Esta etapa silver toma la data de bronze de instrumentos y mejora la calidad de la información eliminando duplicados con un procesamiento incremental. Además este modulo es responsable de generar surrogate pk para el modelo de datos, qué se generarán a partir del nombre del ticket y la fecha de la información.

Inferimos el esquema del json para poder armar el modelo con el esquema qué necesitamos

In [0]:
ruta_origen = "iol_challenge.bronze.raw_ingestion"
ruta_destino = "iol_challenge.silver.deduped_instruments"

In [0]:
from pyspark.sql.functions import from_json, schema_of_json, col

data_sample = spark.read \
    .format("delta") \
    .table(ruta_origen) \
    .filter(col("data_type") == "instrument") \
    .withColumn("schema", schema_of_json(col("data"))) \
    .select("schema").first()[0]

In [0]:
data_sample

'STRUCT<Close: DOUBLE, High: DOUBLE, Low: DOUBLE, Open: DOUBLE, Volume: DOUBLE, date: STRING, simbolo: STRING>'

In [0]:
from pyspark.sql.functions import col, row_number, max as spark_max, length, md5, concat_ws, lit, explode, from_json
from pyspark.sql.window import Window
from delta.tables import DeltaTable

schema = """
STRUCT<simbolo: STRING,
date: STRING, 
Close: DOUBLE, 
High: DOUBLE, 
Low: DOUBLE, 
Open: DOUBLE, 
Volume: DOUBLE>
"""

df_origen_raw = spark.read \
    .format("delta") \
    .table(ruta_origen) \
    .filter(col("data_type") == "instrument") \
    .withColumn("parsed_dict", from_json(col("data"), schema)) \
    .withColumn("sk_instrument", md5(concat_ws(lit("||"), col("parsed_dict.date"), col("parsed_dict.simbolo")))) \
    .select(
        col("sk_instrument"),
        col("parsed_dict.*"), 
        col("dia"),
        col("anio"),
        col("mes"),
        col("timestamp_ejecucion"),
        col("errores_calidad"),
        col("tiene_errores_calidad"),
    )

# Tomamos la última versión ingestada de la transacción preferentemente sin errores de calidad
ventana_dedup = Window.partitionBy("sk_instrument").orderBy(col("tiene_errores_calidad").asc(), col("timestamp_ejecucion").desc())

df_lote_deduplicado = df_origen_raw \
    .withColumn("row_num", row_number().over(ventana_dedup)) \
    .filter(col("row_num") == 1) \
    .drop("row_num") \


if spark.catalog.tableExists(ruta_destino):
    tabla_destino = DeltaTable.forName(spark, ruta_destino)
    
    max_timestamp = tabla_destino.toDF() \
        .select(spark_max("timestamp_ejecucion")) \
        .collect()[0][0]
    
    if max_timestamp is not None:
        df_lote_filtrado = df_lote_deduplicado.filter(col("timestamp_ejecucion") >= max_timestamp)
    else:
        df_lote_filtrado = df_lote_deduplicado

    tabla_destino.alias("target") \
        .merge(
            df_lote_filtrado.alias("source"),
            "target.sk_instrument = source.sk_instrument"
        ) \
        .whenMatchedUpdateAll(
            condition="source.timestamp_ejecucion > target.timestamp_ejecucion"
        ) \
        .whenNotMatchedInsertAll() \
        .execute()

else:
    df_lote_deduplicado.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(ruta_destino)

In [0]:
%sql
SELECT * FROM iol_challenge.silver.deduped_instruments limit 10;


sk_instrument,simbolo,date,Close,High,Low,Open,Volume,dia,anio,mes,timestamp_ejecucion,errores_calidad,tiene_errores_calidad
00019081137a393722407b160ea47200,GDX,2026-06-24,11520.0,11900.0,11420.0,11840.0,11841.0,6,2026,6,2026-08-05T06:40:50.712Z,null,false
0002e9b6bf687d8128cfd1955640ea0c,GOOGL,2026-03-19,7759.99658203125,7789.996568817453,7674.996619470341,7754.99658423355,153198.0,3,2026,3,2026-08-05T06:40:50.712Z,null,false
00032ceece83a4caa4a93b06c32cc991,VALE,2026-03-30,11210.0,11410.0,11150.0,11300.0,39339.0,3,2026,3,2026-08-05T06:40:50.712Z,null,false
0003da4526974f2a362b5aa8bd05b7e4,FXID,2026-07-08,6.900000095367432,6.920000076293945,6.679999828338623,6.75,4251.0,7,2026,7,2026-08-05T06:40:50.712Z,null,false
00043cb3652445f55a27297c8945eb01,NFLX,2026-01-15,2792.5,2852.5,2790.0,2810.0,208259.0,1,2026,1,2026-08-05T06:40:50.712Z,null,false
0004c9a1f3a2ec3292391ce03270ddce,ABNBD,2026-04-15,9.520000457763672,9.579999923706055,9.399999618530273,9.470000267028809,889.0,4,2026,4,2026-08-05T06:40:50.712Z,null,false
0005b2be38bb5e4bd0840ef48b7c709b,CSCO,2026-01-13,22969.86328125,23029.862924126577,22379.866792963647,22379.866792963647,1361.0,1,2026,1,2026-08-05T06:40:50.712Z,null,false
0005be3adfcc02ada1dc5f58b79003a4,HSY,2026-01-30,13779.8671875,13779.8671875,13679.86815130624,13679.86815130624,17.0,1,2026,1,2026-08-05T06:40:50.712Z,null,false
0006b8c239a44ac150c3d16702f0feaa,TGSU2,2026-06-02,9470.0,9520.0,9255.0,9425.0,186644.0,6,2026,6,2026-08-05T06:40:50.712Z,null,false
0006fcc80dc9d3c6ef047262741f1be8,JPM,2026-05-06,31199.912109375,31239.911996694715,30799.913236177887,30899.912954477164,10831.0,5,2026,5,2026-08-05T06:40:50.712Z,null,false


In [0]:
%sql
SELECT date, simbolo, COUNT(*) FROM iol_challenge.silver.deduped_instruments group by date, simbolo
    having count(*)>1;

date,simbolo,COUNT(*)


Revisamos los datos de instrumentos ingestados por mes

In [0]:
%sql
SELECT COUNT(*),  month(date) FROM iol_challenge.silver.deduped_instruments
    GROUP BY month(date)

COUNT(*),month(date)
12204,6
11942,3
12718,7
11856,1
11423,4
11318,5
10294,2
1338,8
